<a href="https://colab.research.google.com/github/Aqillaaprly/rakamin_homework/blob/main/Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Load Data
df = pd.read_csv('/content/drive/MyDrive/rakamin/recruitment_efficiency_improved.csv')
print("Shape:", df.shape)
display(df.head())
print("\nInfo Data:")
df.info()

Shape: (5000, 8)


,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79



Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   recruitment_id         5000 non-null   int64  
 1   department             5000 non-null   object 
 2   job_title              5000 non-null   object 
 3   num_applicants         5000 non-null   int64  
 4   time_to_hire_days      5000 non-null   int64  
 5   cost_per_hire          5000 non-null   float64
 6   source                 5000 non-null   object 
 7   offer_acceptance_rate  5000 non-null   float64
dtypes: float64(2), int64(3), object(3)
memory usage: 312.6+ KB


# Offer Acceptance Rate

In [ ]:
# TRAIN MODEL
def train_model(model, df, target_col, need_scaling=False):
    X = df[['department', 'job_title', 'source']]
    y = df['offer_acceptance_rate']

    categorical = ['department', 'job_title', 'source']

    # Preprocessing
    transformers = [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical)]

    if need_scaling:
        transformers.append(('scale', StandardScaler(with_mean=False), categorical))

    preprocess = ColumnTransformer(transformers)

    # Pipeline model
    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Output
    print("="*60)
    print(f" MODEL: {model.__class__.__name__}")
    print("="*60)
    print(f"Target: {target_col}")
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")
    print("="*60)

    return pipeline


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = train_model(
    RandomForestRegressor(n_estimators=200, random_state=42),
    df,
    target_col="offer_acceptance_rate",
    need_scaling=False
)

 MODEL: RandomForestRegressor
Target: offer_acceptance_rate
MAE : 0.1782
MSE : 0.0436
RMSE: 0.2088
R²  : -0.0406


In [ ]:
df['predicted_offer_acceptance_rate'] = rf_model.predict(
    df[['department', 'job_title', 'source']]
)

df.head()

,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate,predicted_offer_acceptamce_rate,predicted_offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98,0.724997,0.724997
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84,0.627710,0.627710
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83,0.678752,0.678752
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49,0.623344,0.623344
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79,0.649553,0.649553


## Gradient boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = train_model(
    GradientBoostingRegressor(random_state=42),
    df,
    target_col="offer_acceptance_rate"
)

 MODEL: GradientBoostingRegressor
Target: offer_acceptance_rate
MAE : 0.1770
MSE : 0.0429
RMSE: 0.2071
R²  : -0.0234


In [ ]:
df['predicted_offer_acceptance_rate'] = gb_model.predict(
    df[['department', 'job_title', 'source']]
)

df.head()

,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate,predicted_offer_acceptamce_rate,predicted_offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98,0.724997,0.708315
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84,0.627710,0.630405
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83,0.678752,0.649141
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49,0.623344,0.630804
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79,0.649553,0.650156


## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr_model = train_model(
    LinearRegression(),
    df,
    target_col="offer_acceptance_rate"
)

 MODEL: LinearRegression
Target: offer_acceptance_rate
MAE : 0.1765
MSE : 0.0426
RMSE: 0.2064
R²  : -0.0166


In [ ]:
df['predicted_offer_acceptance_rate'] = lr_model.predict(
    df[['department', 'job_title', 'source']]
)

df.head()

,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate,predicted_offer_acceptamce_rate,predicted_offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98,0.724997,0.682541
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84,0.627710,0.638135
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83,0.678752,0.634867
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49,0.623344,0.648855
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79,0.649553,0.649755


## XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = train_model(
    XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        objective='reg:squarederror'
    ),
    df,
    target_col="offer_acceptance_rate"
)

 MODEL: XGBRegressor
Target: offer_acceptance_rate
MAE : 0.1781
MSE : 0.0436
RMSE: 0.2087
R²  : -0.0392


In [ ]:
df['predicted_offer_acceptance_rate'] = xgb_model.predict(
    df[['department', 'job_title', 'source']]
)

df.head()

,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate,predicted_offer_acceptamce_rate,predicted_offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98,0.724997,0.723662
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84,0.627710,0.629804
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83,0.678752,0.679154
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49,0.623344,0.619596
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79,0.649553,0.648521


## SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_svr(df, target_col):
    X = df[['department', 'job_title', 'source']]
    y = df['offer_acceptance_rate']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['department', 'job_title', 'source'])
        ]
    )

    # pipeline
    model = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('svr', SVR(kernel='rbf'))
    ])

    # split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # training
    model.fit(X_train, y_train)

    # prediksi
    y_pred = model.predict(X_test)

    # evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"SVR Results for {target_col}:")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)

    return model


In [ ]:
svr_accept = train_svr(df, "offer_acceptance_rate")

SVR Results for offer_acceptance_rate:
MAE : 0.17928234957335884
RMSE: 0.21078206515895262
R²  : -0.060116625474677665


In [ ]:
df['predicted_offer_acceptance_rate'] = svr_accept.predict(
    df[['department', 'job_title', 'source']]
)

df.head()

,recruitment_id,department,job_title,num_applicants,time_to_hire_days,cost_per_hire,source,offer_acceptance_rate,predicted_offer_acceptamce_rate,predicted_offer_acceptance_rate
0,1,Engineering,Software Engineer,280,24,1400.38,Referral,0.98,0.724997,0.730469
1,2,Sales,Account Executive,262,7,2730.45,LinkedIn,0.84,0.627710,0.609698
2,3,Product,UX Designer,11,76,5286.12,LinkedIn,0.83,0.678752,0.659923
3,4,Engineering,DevOps Engineer,238,26,5255.78,Recruiter,0.49,0.623344,0.590283
4,5,HR,Talent Acquisition,275,37,4987.03,LinkedIn,0.79,0.649553,0.659933


# Time to hire

In [ ]:
# TRAIN MODEL
def train_model(model, df, target_col, need_scaling=False):
    X = df[['department', 'job_title', 'source']]
    y = df['time_to_hire_days']

    categorical = ['department', 'job_title', 'source']

    # Preprocessing
    transformers = [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical)]

    if need_scaling:
        transformers.append(('scale', StandardScaler(with_mean=False), categorical))

    preprocess = ColumnTransformer(transformers)

    # Pipeline model
    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Output
    print("="*60)
    print(f" MODEL: {model.__class__.__name__}")
    print("="*60)
    print(f"Target: {target_col}")
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")
    print("="*60)

    return pipeline


## Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf2_model = train_model(
    RandomForestRegressor(n_estimators=200, random_state=42),
    df,
    target_col="time_to_hire_days",
    need_scaling=False
)

 MODEL: RandomForestRegressor
Target: time_to_hire_days
MAE : 20.3818
MSE : 567.1736
RMSE: 23.8154
R²  : -0.0272


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb2_model = train_model(
    GradientBoostingRegressor(random_state=42),
    df,
    target_col="time_per_hire_days"
)

 MODEL: GradientBoostingRegressor
Target: time_per_hire_days
MAE : 20.2670
MSE : 560.7162
RMSE: 23.6794
R²  : -0.0155


## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr2_model = train_model(
    LinearRegression(),
    df,
    target_col="time_to_hire_days"
)

 MODEL: LinearRegression
Target: time_to_hire_days
MAE : 20.2472
MSE : 559.8264
RMSE: 23.6607
R²  : -0.0139


## XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb2_model = train_model(
    XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        objective='reg:squarederror'
    ),
    df,
    target_col="time_to_hire_days"
)

 MODEL: XGBRegressor
Target: time_to_hire_days
MAE : 20.3800
MSE : 566.9394
RMSE: 23.8105
R²  : -0.0268


## SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_svr(df, target_col):
    X = df[['department', 'job_title', 'source']]
    y = df['time_to_hire_days']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['department', 'job_title', 'source'])
        ]
    )

    # pipeline
    model = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('svr', SVR(kernel='rbf'))
    ])

    # split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # training
    model.fit(X_train, y_train)

    # prediksi
    y_pred = model.predict(X_test)

    # evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"SVR Results for {target_col}:")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)

    return model


In [ ]:
svr_time = train_svr(df, "time_to_hire_days")

SVR Results for time_to_hire_days:
MAE : 20.32731087352157
RMSE: 23.758612846012657
R²  : -0.022296278077184706


# Cost

In [ ]:
# TRAIN MODEL
def train_model(model, df, target_col, need_scaling=False):
    X = df[['department', 'job_title', 'source']]
    y = df['cost_per_hire']

    categorical = ['department', 'job_title', 'source']

    # Preprocessing
    transformers = [('onehot', OneHotEncoder(handle_unknown='ignore'), categorical)]

    if need_scaling:
        transformers.append(('scale', StandardScaler(with_mean=False), categorical))

    preprocess = ColumnTransformer(transformers)

    # Pipeline model
    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Output
    print("="*60)
    print(f" MODEL: {model.__class__.__name__}")
    print("="*60)
    print(f"Target: {target_col}")
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")
    print("="*60)

    return pipeline


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf3_model = train_model(
    RandomForestRegressor(n_estimators=200, random_state=42),
    df,
    target_col="cost_per_hire",
    need_scaling=False
)

 MODEL: RandomForestRegressor
Target: cost_per_hire
MAE : 2372.0367
MSE : 7601619.0855
RMSE: 2757.1034
R²  : -0.0479


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb3_model = train_model(
    GradientBoostingRegressor(random_state=42),
    df,
    target_col="cost_per_hire"
)

 MODEL: GradientBoostingRegressor
Target: cost_per_hire
MAE : 2345.1135
MSE : 7398204.9703
RMSE: 2719.9641
R²  : -0.0199


##

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr3_model = train_model(
    LinearRegression(),
    df,
    target_col="cost_per_hire"
)

 MODEL: LinearRegression
Target: cost_per_hire
MAE : 2335.0259
MSE : 7344259.7815
RMSE: 2710.0295
R²  : -0.0125


## XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb3_model = train_model(
    XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        objective='reg:squarederror'
    ),
    df,
    target_col="cost_per_hire"
)

 MODEL: XGBRegressor
Target: cost_per_hire
MAE : 2372.3430
MSE : 7600285.3637
RMSE: 2756.8615
R²  : -0.0478


## SVr

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_svr(df, target_col):
    X = df[['department', 'job_title', 'source']]
    y = df['cost_per_hire']

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['department', 'job_title', 'source'])
        ]
    )

    # pipeline
    model = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('svr', SVR(kernel='rbf'))
    ])

    # split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # training
    model.fit(X_train, y_train)

    # prediksi
    y_pred = model.predict(X_test)

    # evaluasi
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"SVR Results for {target_col}:")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)

    return model


In [ ]:
svr_cost = train_svr(df, "cost_per_hire")

SVR Results for cost_per_hire:
MAE : 2327.752679790164
RMSE: 2696.241933534599
R²  : -0.002189291179815811
